# 06 — Configuración de BigQuery

Este notebook crea el dataset `chicago_crimes_results` en BigQuery, define las tablas de resultados agregados, y carga la **tabla de hechos completa** (`crimes_fact`) directamente desde GCS — sin pasar por Python.

**Proyecto:** `my-first-project-492901`  
**Dataset BQ:** `chicago_crimes_results`  
**Región:** `US` (multi-región — compatible con GCS us-central1)

### Tablas a crear

| Tabla | Origen | Rol en Power BI |
|---|---|---|
| `crimes_fact` | GCS → BQ Load Job (nativo) | **Tabla de hechos** — modelo estrella |
| `arrests_by_year` | Spark CRUD 1 | Página 1 — tendencia histórica |
| `crimes_by_district` | Modin CRUD 3 | Página 2 — geografía |
| `monthly_trend` | Spark CRUD 3 | Página 3 — temporalidad |
| `hourly_distribution` | Dask CRUD 1 | Página 3 — temporalidad |
| `arrest_rate_by_type` | Dask CRUD 3 | Página 4 — efectividad policial |
| `domestic_trend` | Modin CRUD 1 | Página 5 — violencia doméstica |
| `top_blocks` | Modin CRUD 3 | Página 2 — geografía |
| `severity_distribution` | Dask CRUD 4 | Página 4 — efectividad policial |
| `weekend_analysis` | Spark CRUD 4 | Página 3 — temporalidad |
| `kpis_summary` | KPIs notebook | Todas las páginas (tarjetas KPI) |

### Por qué GCS → BigQuery Load Job es el mejor enfoque para `crimes_fact`

- **Nativo y sin intermediarios:** BigQuery lee los CSVs directamente desde GCS dentro de la infraestructura de Google — sin descargar datos a tu PC ni cargarlos en memoria Python.
- **Velocidad:** Carga 6.4M registros en ~30-60 segundos usando el motor interno de BQ.
- **Sin costo de egress:** La transferencia GCS → BigQuery dentro del mismo proyecto no genera costo de red.
- **Schema enforcement:** BigQuery valida cada fila contra el schema definido durante la carga, rechazando registros malformados automáticamente.

In [1]:
# !pip install google-cloud-bigquery pandas-gbq

In [2]:
from google.cloud import bigquery

PROJECT_ID = 'my-first-project-492901'
DATASET_ID = 'chicago_crimes_results'
LOCATION   = 'US'

client = bigquery.Client(project=PROJECT_ID)
print(f'BigQuery client inicializado — proyecto: {PROJECT_ID}')

C:\Users\Usuario\miniconda3\envs\bigdata\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


BigQuery client inicializado — proyecto: my-first-project-492901


In [3]:
# ── Crear dataset si no existe ───────────────────────────────────────────────
dataset_ref = bigquery.Dataset(f'{PROJECT_ID}.{DATASET_ID}')
dataset_ref.location = LOCATION
dataset_ref.description = 'Resultados del procesamiento distribuido del dataset Chicago Crimes 2001-2026'

dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f'Dataset listo: {PROJECT_ID}.{DATASET_ID}  [{LOCATION}]')

Dataset listo: my-first-project-492901.chicago_crimes_results  [US]


In [4]:
# ── Schemas de las tablas ────────────────────────────────────────────────────
TABLE_SCHEMAS = {
    'arrests_by_year': [
        bigquery.SchemaField('year',              'INTEGER'),
        bigquery.SchemaField('covid_era',         'STRING'),
        bigquery.SchemaField('total_crimes',      'INTEGER'),
        bigquery.SchemaField('total_arrests',     'INTEGER'),
        bigquery.SchemaField('arrest_rate_pct',   'FLOAT'),
    ],
    'crimes_by_district': [
        bigquery.SchemaField('district',          'INTEGER'),
        bigquery.SchemaField('covid_era',         'STRING'),
        bigquery.SchemaField('total_crimes',      'INTEGER'),
        bigquery.SchemaField('total_arrests',     'INTEGER'),
        bigquery.SchemaField('arrest_rate_pct',   'FLOAT'),
        bigquery.SchemaField('distinct_crime_types','INTEGER'),
        bigquery.SchemaField('domestic_incidents','INTEGER'),
    ],
    'monthly_trend': [
        bigquery.SchemaField('year',              'INTEGER'),
        bigquery.SchemaField('covid_era',         'STRING'),
        bigquery.SchemaField('month',             'INTEGER'),
        bigquery.SchemaField('total_crimes',      'INTEGER'),
        bigquery.SchemaField('arrests',           'INTEGER'),
        bigquery.SchemaField('arrest_rate_pct',   'FLOAT'),
    ],
    'hourly_distribution': [
        bigquery.SchemaField('hour_of_day',       'INTEGER'),
        bigquery.SchemaField('covid_era',         'STRING'),
        bigquery.SchemaField('total_crimes',      'INTEGER'),
        bigquery.SchemaField('arrests',           'INTEGER'),
        bigquery.SchemaField('arrest_rate_pct',   'FLOAT'),
    ],
    'arrest_rate_by_type': [
        bigquery.SchemaField('primary_type',      'STRING'),
        bigquery.SchemaField('covid_era',         'STRING'),
        bigquery.SchemaField('total_crimes',      'INTEGER'),
        bigquery.SchemaField('arrest_rate_pct',   'FLOAT'),
    ],
    'domestic_trend': [
        bigquery.SchemaField('year',              'INTEGER'),
        bigquery.SchemaField('covid_era',         'STRING'),
        bigquery.SchemaField('year_quarter',      'STRING'),
        bigquery.SchemaField('domestic_crimes',   'INTEGER'),
        bigquery.SchemaField('total_crimes',      'INTEGER'),
        bigquery.SchemaField('domestic_rate_pct', 'FLOAT'),
    ],
    'top_blocks': [
        bigquery.SchemaField('block',             'STRING'),
        bigquery.SchemaField('total_crimes',      'INTEGER'),
        bigquery.SchemaField('arrests',           'INTEGER'),
        bigquery.SchemaField('domestic_incidents','INTEGER'),
        bigquery.SchemaField('distinct_crime_types','INTEGER'),
        bigquery.SchemaField('arrest_rate_pct',   'FLOAT'),
    ],
    'severity_distribution': [
        bigquery.SchemaField('covid_era',         'STRING'),
        bigquery.SchemaField('severity_category', 'STRING'),
        bigquery.SchemaField('count',             'INTEGER'),
        bigquery.SchemaField('percentage',        'FLOAT'),
    ],
    'weekend_analysis': [
        bigquery.SchemaField('covid_era',         'STRING'),
        bigquery.SchemaField('is_weekend',        'BOOLEAN'),
        bigquery.SchemaField('total_crimes',      'INTEGER'),
        bigquery.SchemaField('total_arrests',     'INTEGER'),
        bigquery.SchemaField('arrest_rate_pct',   'FLOAT'),
        bigquery.SchemaField('pct_of_era_total',  'FLOAT'),
    ],
    'kpis_summary': [
        bigquery.SchemaField('kpi_id',            'INTEGER'),
        bigquery.SchemaField('kpi_name',          'STRING'),
        bigquery.SchemaField('value_numeric',     'FLOAT'),
        bigquery.SchemaField('value_text',        'STRING'),
        bigquery.SchemaField('unit',              'STRING'),
        bigquery.SchemaField('framework',         'STRING'),
    ],
}

print(f'Schemas definidos para {len(TABLE_SCHEMAS)} tablas.')

Schemas definidos para 10 tablas.


In [5]:
# ── Crear tablas vacías ──────────────────────────────────────────────────────
for table_name, schema in TABLE_SCHEMAS.items():
    table_ref = client.dataset(DATASET_ID).table(table_name)
    table     = bigquery.Table(table_ref, schema=schema)
    table     = client.create_table(table, exists_ok=True)
    print(f'  ✓  {DATASET_ID}.{table_name}')

print(f'\n{len(TABLE_SCHEMAS)} tablas listas en BigQuery.')

  ✓  chicago_crimes_results.arrests_by_year


  ✓  chicago_crimes_results.crimes_by_district


  ✓  chicago_crimes_results.monthly_trend


  ✓  chicago_crimes_results.hourly_distribution


  ✓  chicago_crimes_results.arrest_rate_by_type


  ✓  chicago_crimes_results.domestic_trend


  ✓  chicago_crimes_results.top_blocks


  ✓  chicago_crimes_results.severity_distribution


  ✓  chicago_crimes_results.weekend_analysis


  ✓  chicago_crimes_results.kpis_summary

10 tablas listas en BigQuery.


In [6]:
# ── Verificar dataset ────────────────────────────────────────────────────────
tables = list(client.list_tables(DATASET_ID))
print(f'Dataset: {PROJECT_ID}.{DATASET_ID}')
print(f'Tablas creadas: {len(tables)}')
print('─' * 45)
for t in tables:
    print(f'  {t.table_id}')

Dataset: my-first-project-492901.chicago_crimes_results
Tablas creadas: 10
─────────────────────────────────────────────
  arrest_rate_by_type
  arrests_by_year
  crimes_by_district
  domestic_trend
  hourly_distribution
  kpis_summary
  monthly_trend
  severity_distribution
  top_blocks
  weekend_analysis


## Helper: función reutilizable para subir DataFrames a BigQuery

Todos los notebooks de procesamiento importan esta función.

In [7]:
import pandas as pd

def to_bigquery(df: pd.DataFrame, table_name: str, if_exists: str = 'replace') -> None:
    """
    Sube un DataFrame pandas a BigQuery.
    if_exists: 'replace' sobreescribe, 'append' agrega filas.
    """
    destination = f'{PROJECT_ID}.{DATASET_ID}.{table_name}'
    df.to_gbq(
        destination_table=f'{DATASET_ID}.{table_name}',
        project_id=PROJECT_ID,
        if_exists=if_exists,
        progress_bar=False,
    )
    print(f'  → BigQuery: {destination}  ({len(df):,} filas)')

print('Función to_bigquery() lista.')
print('Uso: to_bigquery(df, "arrests_by_year")')

Función to_bigquery() lista.
Uso: to_bigquery(df, "arrests_by_year")


---
## Carga de la Tabla de Hechos: `crimes_fact`

Esta celda carga las 22 columnas completas del dataset Chicago Crimes directamente desde GCS a BigQuery usando un **Load Job nativo** — sin pasar por Python ni memoria RAM local.

La tabla `crimes_fact` es la base del **modelo estrella** en Power BI. Desde ella se derivan todas las dimensiones.

In [8]:
import time

BUCKET_NAME = 'big-data-proyecto-parcial'
GCS_URI     = f'gs://{BUCKET_NAME}/raw/Chicago_Crimes_*.csv'
FACT_TABLE  = 'crimes_fact'

# Schema completo de las 22 columnas — coincide exactamente con los CSVs normalizados
fact_schema = [
    bigquery.SchemaField('unique_key',            'INTEGER',   mode='NULLABLE'),
    bigquery.SchemaField('case_number',           'STRING',    mode='NULLABLE'),
    bigquery.SchemaField('date',                  'TIMESTAMP', mode='NULLABLE'),
    bigquery.SchemaField('block',                 'STRING',    mode='NULLABLE'),
    bigquery.SchemaField('iucr',                  'STRING',    mode='NULLABLE'),
    bigquery.SchemaField('primary_type',          'STRING',    mode='NULLABLE'),
    bigquery.SchemaField('description',           'STRING',    mode='NULLABLE'),
    bigquery.SchemaField('location_description',  'STRING',    mode='NULLABLE'),
    bigquery.SchemaField('arrest',                'BOOLEAN',   mode='NULLABLE'),
    bigquery.SchemaField('domestic',              'BOOLEAN',   mode='NULLABLE'),
    bigquery.SchemaField('beat',                  'INTEGER',   mode='NULLABLE'),
    bigquery.SchemaField('district',              'INTEGER',   mode='NULLABLE'),
    bigquery.SchemaField('ward',                  'INTEGER',   mode='NULLABLE'),
    bigquery.SchemaField('community_area',        'INTEGER',   mode='NULLABLE'),
    bigquery.SchemaField('fbi_code',              'STRING',    mode='NULLABLE'),
    bigquery.SchemaField('x_coordinate',          'INTEGER',   mode='NULLABLE'),
    bigquery.SchemaField('y_coordinate',          'INTEGER',   mode='NULLABLE'),
    bigquery.SchemaField('year',                  'INTEGER',   mode='NULLABLE'),
    bigquery.SchemaField('updated_on',            'TIMESTAMP', mode='NULLABLE'),
    bigquery.SchemaField('latitude',              'FLOAT',     mode='NULLABLE'),
    bigquery.SchemaField('longitude',             'FLOAT',     mode='NULLABLE'),
    bigquery.SchemaField('location',              'STRING',    mode='NULLABLE'),
]

# Configuración del Load Job
job_config = bigquery.LoadJobConfig(
    schema              = fact_schema,
    source_format       = bigquery.SourceFormat.CSV,
    skip_leading_rows   = 1,           # saltar header de cada CSV
    write_disposition   = bigquery.WriteDisposition.WRITE_TRUNCATE,  # sobreescribir si existe
    allow_jagged_rows   = True,        # tolerar filas con columnas faltantes al final
    ignore_unknown_values = True,      # ignorar columnas extra si las hubiera
    max_bad_records     = 1000,        # tolerar hasta 1000 registros malformados
    null_marker         = '',          # string vacío = NULL
)

table_ref = client.dataset(DATASET_ID).table(FACT_TABLE)

print(f'Iniciando carga: {GCS_URI}')
print(f'Destino:         {PROJECT_ID}.{DATASET_ID}.{FACT_TABLE}')
print(f'Modo:            WRITE_TRUNCATE (sobreescribe si existe)')
print('─' * 60)

t0  = time.time()
job = client.load_table_from_uri(GCS_URI, table_ref, job_config=job_config)

# Esperar a que el job termine
job.result()

elapsed = time.time() - t0
table   = client.get_table(table_ref)

print(f'✓ Carga completada en {elapsed:.1f}s')
print(f'  Filas cargadas : {table.num_rows:,}')
print(f'  Columnas       : {len(table.schema)}')
print(f'  Tamaño en BQ   : {table.num_bytes / (1024**2):.1f} MB')
if job.errors:
    print(f'  Errores        : {len(job.errors)}')
    for e in job.errors[:5]:
        print(f'    {e}')

Iniciando carga: gs://big-data-proyecto-parcial/raw/Chicago_Crimes_*.csv
Destino:         my-first-project-492901.chicago_crimes_results.crimes_fact
Modo:            WRITE_TRUNCATE (sobreescribe si existe)
────────────────────────────────────────────────────────────
✓ Carga completada en 14.7s
  Filas cargadas : 8,385,851
  Columnas       : 22
  Tamaño en BQ   : 1715.6 MB


---
## Modelo Estrella para Power BI

Con `crimes_fact` cargada en BigQuery, Power BI puede construir el siguiente modelo estrella. Las dimensiones se crean como **vistas SQL en BigQuery** o directamente en Power Query.

```
                    ┌─────────────────┐
                    │   dim_date      │
                    │─────────────────│
                    │ date (PK)       │
                    │ year            │
                    │ month           │
                    │ day             │
                    │ quarter         │
                    │ day_of_week     │
                    │ is_weekend      │
                    └────────┬────────┘
                             │
┌──────────────┐    ┌────────▼────────┐    ┌──────────────────┐
│ dim_location │    │  crimes_fact    │    │  dim_crime_type  │
│──────────────│    │─────────────────│    │──────────────────│
│ district(PK) │◄───│ unique_key (PK) │───►│ iucr (PK)        │
│ ward         │    │ date            │    │ primary_type     │
│ community_   │    │ iucr            │    │ description      │
│   area       │    │ district        │    │ fbi_code         │
│ beat         │    │ beat            │    │ severity (HIGH/  │
│ block        │    │ block           │    │  MEDIUM/LOW)     │
│ latitude     │    │ arrest          │    └──────────────────┘
│ longitude    │    │ domestic        │
└──────────────┘    │ latitude        │    ┌──────────────────┐
                    │ longitude       │    │  dim_location_   │
                    │ ...             │───►│    description   │
                    └─────────────────┘    │──────────────────│
                                           │ location_desc(PK)│
                                           │ location_group   │
                                           └──────────────────┘
```

### Cómo crear las dimensiones en Power BI (Power Query)

1. **Conectar a BigQuery:** Inicio → Obtener datos → Google BigQuery → proyecto `my-first-project-492901`
2. **Importar `crimes_fact`** como tabla base
3. **Crear dimensiones** duplicando la consulta y agrupando:
   - `dim_date`: extraer columnas de `date` con `Table.TransformColumns`
   - `dim_crime_type`: `Table.Distinct` sobre `{iucr, primary_type, description, fbi_code}`
   - `dim_location`: `Table.Distinct` sobre `{district, ward, community_area, beat}`
   - `dim_location_description`: `Table.Distinct` sobre `{location_description}`
4. **Definir relaciones** en la vista Modelo de Power BI usando `unique_key` como FK